In [1]:
import ollama
import os

import faiss
import numpy as np

In [2]:

def load_documents(data_dir):

    documents = []

    for file_name in os.listdir(data_dir):

        if file_name.endswith(".txt"):

            file_path = os.path.join( data_dir, file_name )

            with open(file_path, "r", encoding="utf-8" ) as file:

                text = file.read()

            documents.append({ "source": file_name, "text": text })

            print( "Document loaded:", file_name)

    return documents


In [3]:
documents = load_documents("Agriculture")

print( "\nTotal documents:",len(documents))

Document loaded: crop_production.txt
Document loaded: government_schemes.txt
Document loaded: irrigation_management.txt
Document loaded: pest_and_disease.txt
Document loaded: soil_health.txt

Total documents: 5


In [4]:
def chunk_documents(documents):

    chunk_size = 800
    chunk_overlap = 100

    chunks = []

    for document in documents:

        text = document["text"]
        source = document["source"]

        start = 0

        while start < len(text):

            end = start + chunk_size

            chunk = text[start:end]

            chunks.append({ "source": source, "text": chunk})

            start = end - chunk_overlap

    return chunks


In [5]:
chunks = chunk_documents(documents)

print("\nTotal chunks:",len(chunks))


Total chunks: 21


In [6]:

def create_embeddings(chunks):

    embeddings = []

    for chunk in chunks:

        response = ollama.embeddings(
            model="nomic-embed-text",
            prompt=chunk["text"]
        )

        embeddings.append(
            response["embedding"]
        )

    return np.array(
        embeddings,
        dtype="float32"
    )


In [7]:
chunk_embeddings = create_embeddings(chunks)

print( "\nEmbedding shape:",chunk_embeddings.shape)


Embedding shape: (21, 768)


In [8]:
chunk_embeddings[0]

array([ 4.77529347e-01,  1.09547779e-01, -4.00623512e+00,  6.82173014e-01,
       -2.68818319e-01, -1.13222227e-01,  4.94155407e-01, -9.55540955e-01,
        4.30278301e-01,  5.03239572e-01,  1.03700709e+00, -1.97496846e-01,
        2.27097368e+00,  1.72800481e-01,  2.26047084e-01,  4.78792042e-01,
       -1.20197606e+00, -1.72054451e-02,  9.94309559e-02,  1.28658164e+00,
        2.45406210e-01,  7.01840937e-01,  5.23635268e-01, -4.06083405e-01,
        1.08958828e+00,  3.98695648e-01, -6.21982455e-01,  3.03125352e-01,
        4.20348763e-01, -6.12011179e-02,  6.42658472e-02, -5.58568895e-01,
       -5.43992937e-01, -6.21104166e-02, -7.22065032e-01, -1.50892484e+00,
        1.03539777e+00, -8.95117670e-02,  1.41851199e+00, -7.51350522e-01,
        1.19953346e+00, -1.66148007e-01,  1.03910363e+00, -4.43237603e-01,
       -2.62344778e-02,  7.17770994e-01,  1.26500344e+00,  7.03817070e-01,
        1.36610463e-01, -9.35981810e-01,  4.36136007e-01, -1.40915275e-01,
        2.77034700e-01, -

In [9]:

def create_faiss_index(embeddings):

    dimension = embeddings.shape[1]
    
    index = faiss.IndexFlatL2( dimension)
    
    index.add(embeddings)
    
    return index

In [10]:

index = create_faiss_index(chunk_embeddings)

print("Vectors stored in FAISS:",index.ntotal)

Vectors stored in FAISS: 21


In [11]:

question = input("\nEnter your question: ")

print( "\nQuestion:",question)


Enter your question:  "What happens when soil pH is below 5.5?"



Question: "What happens when soil pH is below 5.5?"


In [27]:

def embed_question(question):
    response = ollama.embeddings(
        model="nomic-embed-text",
        prompt=question
    )

    question_embedding = np.array(
        [response["embedding"]],
        dtype="float32"
    )

    return question_embedding


In [28]:
question_embedding = embed_question(question)

print("\nQuestion embedding shape:",question_embedding.shape)


Question embedding shape: (1, 768)


In [29]:

def search_faiss( question_embedding, index, k= 5):

    distances, indices = index.search( question_embedding, k)

    return distances[0], indices[0]


In [30]:

distances, indices = search_faiss( question_embedding, index, k=5)

print( "\nDistances:", distances)
print( "Chunk numbers:", indices)


Distances: [257.7716  265.29312 299.6548  319.15994 342.42908]
Chunk numbers: [17 18 19 20 16]


In [31]:

def retrieve_chunks( indices, chunks):

    retrieved_chunks = []

    for i in indices:
        retrieved_chunks.append( chunks[i] )
    return retrieved_chunks


In [32]:
retrieved_chunks = retrieve_chunks(indices, chunks)

print( "\nNumber of retrieved chunks:", len(retrieved_chunks))


Number of retrieved chunks: 5


In [33]:

for i, chunk in enumerate( retrieved_chunks):

    print( f"\n===== RETRIEVED CHUNK {i + 1} =====" )

    print( "Source:", chunk["source"])

    print( "Distance:", distances[i])

    print("\nText:")

    print(chunk["text"] )


===== RETRIEVED CHUNK 1 =====
Source: soil_health.txt
Distance: 257.7716

Text:
SOIL HEALTH AND FERTILIZER MANAGEMENT GUIDE 

1. DIAGNOSTIC PARAMETERS OF AGRICULTURAL SOILS
Evaluating soil fertility requires analyzing three primary dimensions:
- Physical Properties: Soil texture (ratio of sand, silt, and clay) determines water retention capacity and aeration. Ideal loam soils possess balanced drainage. Bulk density should remain below 1.4 g/cm³ to prevent root restriction.
- Chemical Properties: 
  * Soil pH: The optimal range for maximum nutrient availability is 6.5 to 7.5. Soils with pH < 5.5 suffer from aluminum toxicity and phosphorus fixation, requiring agricultural lime amendments. Saline-sodic soils with pH > 8.5 require gypsum treatment.
  * Electrical Conductivity (EC): Must remain below 1.0 dS/m. Values exceeding 2.0 dS/m cause osmotic stress, limiting water

===== RETRIEVED CHUNK 2 =====
Source: soil_health.txt
Distance: 265.29312

Text:
ity (EC): Must remain below 1.0 dS/m

In [34]:

def create_context( retrieved_chunks ):

    context = ""

    for chunk in retrieved_chunks:

        context += f"""
Source: {chunk["source"]}

{chunk["text"]}

"""

    return context


In [35]:

context = create_context( retrieved_chunks)

print("\n===== CONTEXT =====")
print(context)



===== CONTEXT =====

Source: soil_health.txt

SOIL HEALTH AND FERTILIZER MANAGEMENT GUIDE 

1. DIAGNOSTIC PARAMETERS OF AGRICULTURAL SOILS
Evaluating soil fertility requires analyzing three primary dimensions:
- Physical Properties: Soil texture (ratio of sand, silt, and clay) determines water retention capacity and aeration. Ideal loam soils possess balanced drainage. Bulk density should remain below 1.4 g/cm³ to prevent root restriction.
- Chemical Properties: 
  * Soil pH: The optimal range for maximum nutrient availability is 6.5 to 7.5. Soils with pH < 5.5 suffer from aluminum toxicity and phosphorus fixation, requiring agricultural lime amendments. Saline-sodic soils with pH > 8.5 require gypsum treatment.
  * Electrical Conductivity (EC): Must remain below 1.0 dS/m. Values exceeding 2.0 dS/m cause osmotic stress, limiting water


Source: soil_health.txt

ity (EC): Must remain below 1.0 dS/m. Values exceeding 2.0 dS/m cause osmotic stress, limiting water uptake by roots.
  * Soi

In [36]:

def create_prompt( question, context ):

    prompt = f"""
You are an agricultural information assistant.

Answer the question using ONLY the information
provided in the context.

Do not use outside knowledge.

If the answer is not available in the context,
say:

"I could not find the answer in the provided documents."

Give a clear and concise answer.

Context:
{context}

Question:
{question}

Answer:
"""

    return prompt


In [37]:

prompt = create_prompt( question, context)

In [38]:

def generate_answer(prompt):

    response = ollama.chat(
 
        model="llama3.2",

        messages=[
            { "role": "user", "content": prompt }
        ],

        options={ "temperature": 0}
    )

    return response["message"]["content"]


In [39]:

answer = generate_answer( prompt)

In [40]:

print( "FINAL RAG ANSWER")

print(answer)


FINAL RAG ANSWER
Soils with pH < 5.5 suffer from aluminum toxicity and phosphorus fixation, requiring agricultural lime amendments.


In [26]:

print( "\nSources used:")

for chunk in retrieved_chunks:

    print( "-", chunk["source"] )


Sources used:
- soil_health.txt
- soil_health.txt
- soil_health.txt
- soil_health.txt
- pest_and_disease.txt
